# Team 2 — Feature Extraction
### Member: Khadija
### Project: Poaching Threat Detection | SDG Goal 15 — Life on Land
---
**Responsibilities:**
- Extract Tonnetz (tonal network) features (12 features)
- Run the full feature extraction pipeline over all 2025 files
- Save final `audio_features.csv` for Team 3 (Model Building)
- Print summary report and verify output shape

**Note:** Run Ranjan, Bharat and Aasim notebooks first


In [ ]:
# ============================================================
# SETUP — Run this if executing notebook standalone
# ============================================================

import os
import numpy as np
import pandas as pd
import librosa
from tqdm import tqdm

SAMPLE_RATE = 44100
N_MFCC      = 40
N_MELS      = 128
N_FFT       = 2048
HOP_LENGTH  = 512

CSV_OUTPUT = 'metadata.csv'
df = pd.read_csv(CSV_OUTPUT)
print(f'Loaded {len(df)} files')


In [ ]:
# ============================================================
# CELL 6 — TONNETZ FEATURE EXTRACTION
# Khadija — Tonal Network Features
# ============================================================

def extract_tonnetz_features(audio, sr=SAMPLE_RATE):
    """
    Extracts Tonnetz (tonal network) features.
    Tonnetz represents the harmonic relationships in a sound.
    Wildlife sounds (bird chirping, frog calls) have rich harmonic
    content while mechanical threats have poor or no harmonic structure.

    Features extracted:
      - Tonnetz (mean + std of 6 dimensions) → 12 features

    Note: Silence check is applied before computing tonnetz
    to avoid NaN values on near-silent clips.
    """
    features = {}

    # Extract harmonic component
    harmonic = librosa.effects.harmonic(audio)

    # Only compute tonnetz if audio is not silent
    if np.max(np.abs(harmonic)) > 0.01:
        tonnetz = librosa.feature.tonnetz(y=harmonic, sr=sr)
    else:
        tonnetz = np.zeros((6, 1))  # silent clip → zero tonnetz

    for i in range(tonnetz.shape[0]):
        features[f'tonnetz_{i+1}_mean'] = np.mean(tonnetz[i])
        features[f'tonnetz_{i+1}_std']  = np.std(tonnetz[i])

    return features

print('extract_tonnetz_features() function defined!')
print('Tonnetz features covered by Khadija: 6 dimensions × mean + std = 12 features')


In [ ]:
# ============================================================
# CELL 7 — FULL FEATURE EXTRACTION PIPELINE
# Khadija — Runs all features over all 2025 files
# Combines Bharat's + Aasim's + Khadija's features
# ============================================================

# Import all feature functions (run other notebooks first)
# Or define inline for standalone execution

def extract_all_features(file_path, sr=SAMPLE_RATE):
    try:
        audio, _ = librosa.load(file_path, sr=sr, mono=True)
        features = {}

        # ── MFCCs (Bharat) ─────────────────────────────────
        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH)
        for i in range(N_MFCC):
            features[f'mfcc_{i+1}_mean'] = np.mean(mfcc[i])
            features[f'mfcc_{i+1}_std']  = np.std(mfcc[i])
        delta_mfcc = librosa.feature.delta(mfcc)
        for i in range(N_MFCC):
            features[f'delta_mfcc_{i+1}_mean'] = np.mean(delta_mfcc[i])
            features[f'delta_mfcc_{i+1}_std']  = np.std(delta_mfcc[i])
        delta2_mfcc = librosa.feature.delta(mfcc, order=2)
        for i in range(N_MFCC):
            features[f'delta2_mfcc_{i+1}_mean'] = np.mean(delta2_mfcc[i])
            features[f'delta2_mfcc_{i+1}_std']  = np.std(delta2_mfcc[i])
        mel = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        for i in range(N_MELS):
            features[f'mel_{i+1}_mean'] = np.mean(mel_db[i])
            features[f'mel_{i+1}_std']  = np.std(mel_db[i])
        chroma = librosa.feature.chroma_stft(y=audio, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH)
        for i in range(chroma.shape[0]):
            features[f'chroma_{i+1}_mean'] = np.mean(chroma[i])
            features[f'chroma_{i+1}_std']  = np.std(chroma[i])

        # ── Spectral Features (Aasim) ───────────────────────
        centroid = librosa.feature.spectral_centroid(y=audio, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH)
        features['spectral_centroid_mean'] = np.mean(centroid)
        features['spectral_centroid_std']  = np.std(centroid)
        bandwidth = librosa.feature.spectral_bandwidth(y=audio, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH)
        features['spectral_bandwidth_mean'] = np.mean(bandwidth)
        features['spectral_bandwidth_std']  = np.std(bandwidth)
        rolloff = librosa.feature.spectral_rolloff(y=audio, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH)
        features['spectral_rolloff_mean'] = np.mean(rolloff)
        features['spectral_rolloff_std']  = np.std(rolloff)
        contrast = librosa.feature.spectral_contrast(y=audio, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH)
        for i in range(contrast.shape[0]):
            features[f'spectral_contrast_{i+1}_mean'] = np.mean(contrast[i])
            features[f'spectral_contrast_{i+1}_std']  = np.std(contrast[i])
        zcr = librosa.feature.zero_crossing_rate(y=audio, hop_length=HOP_LENGTH)
        features['zcr_mean'] = np.mean(zcr)
        features['zcr_std']  = np.std(zcr)
        rms = librosa.feature.rms(y=audio, hop_length=HOP_LENGTH)
        features['rms_mean'] = np.mean(rms)
        features['rms_std']  = np.std(rms)

        # ── Tonnetz (Khadija) ───────────────────────────────
        harmonic = librosa.effects.harmonic(audio)
        if np.max(np.abs(harmonic)) > 0.01:
            tonnetz = librosa.feature.tonnetz(y=harmonic, sr=sr)
        else:
            tonnetz = np.zeros((6, 1))
        for i in range(tonnetz.shape[0]):
            features[f'tonnetz_{i+1}_mean'] = np.mean(tonnetz[i])
            features[f'tonnetz_{i+1}_std']  = np.std(tonnetz[i])

        return features

    except Exception as e:
        print(f'\n  [ERROR] {os.path.basename(file_path)}: {e}')
        return None

print('Full pipeline function defined!')
print('\nExtracting features from all 2025 files...')

all_features = []
failed = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    feats = extract_all_features(row['processed_path'])
    if feats is not None:
        feats['filename']       = row['filename']
        feats['original_class'] = row['original_class']
        feats['label']          = row['label']
        feats['label_encoded']  = row['label_encoded']
        all_features.append(feats)
    else:
        failed.append(row['filename'])

features_df = pd.DataFrame(all_features)
print(f'\nFeature extraction complete!')
print(f'Shape: {features_df.shape}')
print(f'Failed files: {len(failed)}')


In [ ]:
# ============================================================
# CELL 8 — SAVE FEATURES CSV & SUMMARY REPORT
# Khadija — Save output for Team 3 (Model Building)
# ============================================================

# Save to CSV
features_df.to_csv('audio_features.csv', index=False)
print('Features saved to audio_features.csv')

# Summary report
print('\n' + '='*50)
print('   FEATURE EXTRACTION COMPLETE — FSC22')
print('='*50)
print(f'  Total files processed : {len(features_df)}')
print(f'  Total features        : {features_df.shape[1] - 4}')  # minus 4 metadata cols
print(f'  Failed files          : {len(failed)}')
print(f'  NaN values            : {features_df.isnull().sum().sum()}')
print('='*50)

print('\nLabel distribution:')
print(features_df['label'].value_counts())

print('\nFeature breakdown:')
print('  MFCCs (40×2)           =  80 features  [Bharat]')
print('  Delta MFCCs (40×2)     =  80 features  [Bharat]')
print('  Delta2 MFCCs (40×2)    =  80 features  [Bharat]')
print('  Mel Spectrogram (128×2)= 256 features  [Bharat]')
print('  Chroma STFT (12×2)     =  24 features  [Bharat]')
print('  Spectral Features      =  24 features  [Aasim]')
print('  Tonnetz (6×2)          =  12 features  [Khadija]')
print('  ─────────────────────────────────────────────')
print('  TOTAL                  = 556 features')
print('='*50)
print('\nHandoff to Team 3 (Model Building) — READY')
print('Team 3 reads: audio_features.csv')
